# Tutorial 5. ML interatomic potentials for simulations of catalytic surfaces

## IMPORTANT

In order to run this notebook you need to load the UMA model (https://huggingface.co/facebook/UMA.
) on https://drive.google.com/file/d/1pw3jkPMWL6CLmpRGhBPdvzoLn4MzpTEX/view
## Goal of the notebook

The goal of this example is to demonstrate how it is possible to model a surface with an adsorbate and optimize the geometry using machine-learning potential (UMA)

## Description of the method

UMA is a machine-learning interatomic potential trained on large datasets of DFT calculations for catalytic systems. It predicts energies and forces for atomic structures, allowing geometry optimization similar to DFT but at significantly lower computational cost.

UMA github: https://github.com/facebookresearch/fairchem

UMA webpage: https://huggingface.co/facebook/UMA

UMA article: Wood, B. M.; Dzamba, M.; Fu, X.; Gao, M.; Shuaibi, M.; Barroso-Luque, L.; Abdelmaqsoud, K.; Gharakhanyan, V.; Kitchin, J. R.; Levine, D. S.; Michel, K.; Sriram, A.; Cohen, T.; Das, A.; Rizvi, A.; Sahoo, S. J.; Ulissi, Z. W.; Zitnick, C. L.UMA: A Family of Universal Models for Atoms. arXiv 2026, arXiv:2506.23971. [https://arxiv.org/abs/2506.23971](https://arxiv.org/abs/2506.23971).


## Outline of tutorial

STEP 0. Login to Hugging Face

STEP 1. Build a Pt(111) surface with OH adsorbates.

STEP 2. Load the UMA model and run the geometry optimization.



## Installation of packages

First thing first, we need to install the packages and import the neccessary functions.

We import the necessary packages:

ASE https://ase-lib.org/

fairchem https://fair-chem.github.io/

In [2]:
##-ASE
from ase.io import read, write
from ase.constraints import FixAtoms
from ase.optimize import BFGS
from ase import Atoms
from ase.visualize import view

##-Fairchem
from fairchem.core import pretrained_mlip, FAIRChemCalculator

##-General
import os
import torch
import numpy as np

## STEP 1. Build a Pt(111) surface with OH adsorbates

Now we load the UMA model using get_predict_unit function and selecting the model we want. Next we get the structure of the slab and define it as an atoms object. For defining the atoms object we use trajectory[-1] which gives the last configuration of the optimization process. Lastly we make a directory and set it as the output for trajectory files.

Load the UMA model from [this link](https://https://drive.google.com/file/d/1pw3jkPMWL6CLmpRGhBPdvzoLn4MzpTEX/view) and upload it to the same folder where this tutorial is placed.

In [3]:
%%capture
##-Load UMA model
model_path="uma-s-1p1.pt"

##-Downloading the structure
!wget https://raw.githubusercontent.com/doublelayer/test_models/main/Pt-w/Pt111OH_3x4x1.xyz

##-Defining atoms object
structure_path = 'Pt111OH_3x4x1.xyz'
trajectory = read(structure_path, index=':')
atoms = trajectory[-1]

##-Make a directory for data, and set the trajectory file output to that directory
os.makedirs("data", exist_ok=True)
output_traj_path = "data/Pt_uma_relax1.traj"

## STEP 2. Load the UMA model and run the geometry optimization

Now we can costumize the calculation object. We chose to set the PBC of the system as periodic in all directions. Next we have to attatch the previously loaded UMA as a calculator to the atoms object. Before choosing the optimizer and running the calculations, we fix the surface atoms inorder to speed up the calculations. Finally we set the optimizer and run the optimization.

In [4]:
##-Setting PBC
atoms.pbc = [True, True, True]  # or [False, False, False]

##-Creating ASE calculator
calc = FAIRChemCalculator.from_model_checkpoint(
    model_path,
    task_name="oc20",
    device='cpu',
)

##-Attatching the calculator to atoms object
atoms.calc = calc
##-Adding constraints to atoms
cons = FixAtoms(indices=[atom.index for atom in atoms if atom.tag == 0])
atoms.set_constraint(cons)

##-Run the optimization using BFGS
dyn = BFGS(atoms, trajectory=output_traj_path)
dyn.run(fmax=0.1, steps=100)

      Step     Time          Energy          fmax
BFGS:    0 20:17:05     -845.061199        0.758196
BFGS:    1 20:17:17     -845.548153        0.673824
BFGS:    2 20:17:27     -847.388393        0.552788
BFGS:    3 20:17:39     -847.645717        0.469640
BFGS:    4 20:17:50     -847.945370        0.384136
BFGS:    5 20:18:01     -848.097950        0.383823
BFGS:    6 20:18:13     -848.265720        0.439405
BFGS:    7 20:18:24     -848.477238        0.566349
BFGS:    8 20:18:35     -848.778095        0.549069
BFGS:    9 20:18:46     -849.161976        0.627670
BFGS:   10 20:18:57     -849.655399        0.664738
BFGS:   11 20:19:07     -850.193951        0.898495
BFGS:   12 20:19:17     -850.641971        0.962640
BFGS:   13 20:19:28     -851.038929        0.802424
BFGS:   14 20:19:39     -851.398899        0.451592
BFGS:   15 20:19:51     -851.575283        0.314681
BFGS:   16 20:20:01     -851.662823        0.218512
BFGS:   17 20:20:13     -851.735676        0.245864
BFGS:   18 20:

np.True_

In [5]:
view(atoms, viewer='x3d')

## Concluding remarks

In this tutorial we used UMA to model a Pt(111) surface with OH groups. A pretrained model was used to calculate forces and energies to optimize the geometry of Pt(111) with OH adsorbates.

This tutorial demonstrates that the use of machine-learning interatomic potentials of UMA can greatly reduce the optimization time compared to the usual DFT computational time.

After completing this tutorial we can:
* Loadthe UMA machine-learning interatomic potentials
* Run an optimization with UMA as the calculator and BFGS as the optimizer